In [12]:
import pandas as pd
import numpy as np
import re
import html
from collections import Counter

In [3]:
import pandas as pd

se_tags = {
    "java", "c#", "javascript", "c++", "python", "android", "ios", "sql",
    "git", "oop", "multithreading", "design-patterns", "architecture",
    "database-design", "unit-testing", "version-control", "api",
    "web-services", "debugging", "mvc"
}

net_tags = {
    "networking", "linux", "ubuntu", "security", "bash", "apache", "http",
    "ssh", "sockets", "dns", "ssl", "centos", "nginx", "tcp", "routing",
    "wireless-networking", "ftp", "proxy", "active-directory", "amazon-ec2"
}

ai_tags = {
    "opencv",
    "image-processing",
    "r",
    "matlab",
    "algorithm",
    "probability",
    "linear-algebra",
    "matrices",
    "vector",
    "data-structures"
}







In [3]:
se = pd.read_csv("train_se_clean.csv")
net = pd.read_csv("train_net_clean.csv")
ai = pd.read_csv("train_ai_clean.csv")

In [5]:
print(se.columns)
print(net.columns)
print(ai.columns)



Index(['Id', 'Title', 'Body', 'Tags'], dtype='str')
Index(['Id', 'Title', 'Body', 'Tags'], dtype='str')
Index(['Id', 'Title', 'Body', 'Tags'], dtype='str')


In [9]:
se = se[["Title", "Tags"]].copy()
net = net[["Title", "Tags"]].copy()
ai = ai[["Title", "Tags"]].copy()

In [10]:
df = pd.concat([se, net, ai], ignore_index=True)
print(df.shape)

(2769583, 2)


In [ ]:
allowed_tags = se_tags.union(net_tags).union(ai_tags)
print("عدد التاغات المسموحة:", len(allowed_tags))

df["Tags"] = df["Tags"].fillna("").astype(str).str.strip()

df["Tags"] = df["Tags"].apply(
    lambda x: " ".join([tag for tag in x.split() if tag in allowed_tags])
)

df = df[df["Tags"] != ""].copy()

print(df.shape)


عدد التاغات المسموحة: 50
(2769583, 2)
                                               Title        Tags
0      How do I replace special characters in a URL?          c#
1               How to modify whois contact details?         api
2             How to fetch an XML feed using asp.net          c#
3            .NET library for generating javascript?  javascript
4  SQL Server : procedure call, inline concatenat...         sql


In [13]:
df.to_csv("multilabel_tags.csv", index=False)

In [4]:
import re
import html

NORMALIZATION_MAP = {
    "c sharp": "c#",
    "c-sharp": "c#",
    "c plus plus": "c++",
    "cpp": "c++",
    "js": "javascript",
    "nodejs": "node.js",
    "asp net": "asp.net",
    "ms sql": "sql server",
    "mssql": "sql server",
    "postgres": "postgresql",
}

def normalize_terms(text):
    for src, tgt in NORMALIZATION_MAP.items():
        pattern = r"\b" + re.escape(src) + r"\b"
        text = re.sub(pattern, tgt, text)
    return text

def clean_technical_title(text):
    text = str(text)

    # فك HTML
    text = html.unescape(text)

    # lowercase
    text = text.lower()

    # normalization (مهم جداً)
    text = normalize_terms(text)

    # حذف الروابط
    text = re.sub(r"http\S+|www\S+", " ", text)

    # حذف HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # الحفاظ على الرموز التقنية
    text = re.sub(r"[^a-z0-9\s\#\+\.\-_/]", " ", text)

    # تنظيف المسافات
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [5]:
df = pd.read_csv("multilabel_tags.csv")
# 🔥 3) preprocessing هنا
df["Title"] = df["Title"].apply(clean_technical_title)

# 4) حذف الفارغ
df = df[(df["Title"] != "") & (df["Tags"] != "")].copy()

# 5) تحويل Tags
df["Tags"] = df["Tags"].apply(lambda x: x.split())

In [8]:
df_merged = (
    df.groupby("Title", as_index=False)["Tags"]
    .apply(lambda tag_lists: sorted(set(tag for tags in tag_lists for tag in tags)))
)

# إرجاع Tags إلى نص مفصول بمسافة
df_merged["Tags"] = df_merged["Tags"].apply(lambda tags: " ".join(tags))

# حفظ النتيجة
df_merged.to_csv("multilabel_tags_merged.csv", index=False)

# فحص
print("قبل الدمج:", df.shape)
print("بعد الدمج:", df_merged.shape)
print(df_merged.head(10))

قبل الدمج: (2769564, 2)
بعد الدمج: (1866458, 2)
                                               Title        Tags
0                # + items .append is not a function  javascript
1  # - how to parallel code that lock several obj...          c#
2                  # . what do and # do in this code  javascript
3                  # .dialog is not a function error  javascript
4  # .dialog is not a function error after using ...  javascript
5            # /bin/bash - no such file or directory        bash
6  # /usr/bin/env interpreter arguments -- portab...       linux
7  # /usr/bin/env python getting command not foun...      python
8           # /usr/bin/env ruby is not found in cron        bash
9  # can not been used as separator of fields for...           r


In [9]:
df_merged["num_tags"] = df_merged["Tags"].str.split().apply(len)
print(df_merged["num_tags"].value_counts().sort_index())

num_tags
1    1633206
2     213227
3      18670
4       1282
5         67
6          4
7          2
Name: count, dtype: int64


In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

# 1) قراءة الملف بعد الدمج
df = pd.read_csv("multilabel_tags_merged.csv")

In [2]:
df["Title"] = df["Title"].fillna("").astype(str).str.strip()
df["Tags"] = df["Tags"].fillna("").astype(str).str.strip()

df = df[(df["Title"] != "") & (df["Tags"] != "")].copy()

# 3) تحويل Tags إلى قائمة
df["Tags"] = df["Tags"].apply(lambda x: x.split())

# 4) فصل X و y
X = df["Title"]
y = df["Tags"]

In [3]:
# 5) تقسيم البيانات
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)

In [4]:
# 6) MultiLabelBinarizer
mlb = MultiLabelBinarizer()

y_train_bin = mlb.fit_transform(y_train)
y_val_bin = mlb.transform(y_val)
y_test_bin = mlb.transform(y_test)

# 7) حفظ الـ binarizer
joblib.dump(mlb, "multilabel_binarizer.pkl")

['multilabel_binarizer.pkl']

In [15]:
print("Train:", len(X_train))
print("Val:", len(X_val))
print("Test:", len(X_test))


Train: 1493166
Val: 186646
Test: 186646


In [16]:
print("عدد التاغات:", len(mlb.classes_))
print("شكل y_train_bin:", y_train_bin.shape)
print("التاغات:", mlb.classes_)

عدد التاغات: 50
شكل y_train_bin: (1493166, 50)
التاغات: ['active-directory' 'algorithm' 'amazon-ec2' 'android' 'apache' 'api'
 'architecture' 'bash' 'c#' 'c++' 'centos' 'data-structures'
 'database-design' 'debugging' 'design-patterns' 'dns' 'ftp' 'git' 'http'
 'image-processing' 'ios' 'java' 'javascript' 'linear-algebra' 'linux'
 'matlab' 'matrices' 'multithreading' 'mvc' 'networking' 'nginx' 'oop'
 'opencv' 'probability' 'proxy' 'python' 'r' 'routing' 'security'
 'sockets' 'sql' 'ssh' 'ssl' 'tcp' 'ubuntu' 'unit-testing' 'vector'
 'version-control' 'web-services' 'wireless-networking']


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier

# 1) TF-IDF
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.9,
    max_features=150000,
    sublinear_tf=True
)

#  fit فقط على train
X_train_vec = vectorizer.fit_transform(X_train)

# transform للباقي
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

print("شكل X_train:", X_train_vec.shape)

شكل X_train: (1493166, 150000)


In [6]:
import joblib

joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']

In [7]:
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score



model = OneVsRestClassifier(
    LogisticRegression(
        solver="saga",
        max_iter=150,
        random_state=42,
        n_jobs=-1
    ),
    n_jobs=-1
)

model.fit(X_train_vec, y_train_bin)




,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...solver='saga')
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",-1
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=No

# 2) Get scores instead of binary predictions


In [8]:
val_scores = model.predict_proba(X_val_vec)

# If returned as list of arrays, convert to (n_samples, n_labels)
if isinstance(val_scores, list):
    val_scores = np.array([p[:, 1] for p in val_scores]).T


In [9]:
def top_k_binary_predictions(y_scores, k):
    y_pred = np.zeros_like(y_scores, dtype=int)
    topk_idx = np.argsort(-y_scores, axis=1)[:, :k]

    for i in range(y_scores.shape[0]):
        y_pred[i, topk_idx[i]] = 1

    return y_pred

In [10]:
def evaluate_at_k(y_true, y_scores, k):
    y_pred_k = top_k_binary_predictions(y_scores, k)

    precision = precision_score(y_true, y_pred_k, average="micro", zero_division=0)
    recall = recall_score(y_true, y_pred_k, average="micro", zero_division=0)
    f1 = f1_score(y_true, y_pred_k, average="micro", zero_division=0)

    return precision, recall, f1

In [13]:
for k in [1, 2, 3,4]:
    p, r, f = evaluate_at_k(y_val_bin, val_scores, k)

    print(f"\n===== Top-{k} Metrics =====")
    print(f"Precision@{k}: {p:.4f}")
    print(f"Recall@{k}:    {r:.4f}")
    print(f"F1@{k}:        {f:.4f}")



===== Top-1 Metrics =====
Precision@1: 0.7674
Recall@1:    0.6758
F1@1:        0.7187

===== Top-2 Metrics =====
Precision@2: 0.4633
Recall@2:    0.8160
F1@2:        0.5910

===== Top-3 Metrics =====
Precision@3: 0.3294
Recall@3:    0.8704
F1@3:        0.4780

===== Top-4 Metrics =====
Precision@4: 0.2560
Recall@4:    0.9019
F1@4:        0.3988


In [14]:
joblib.dump(model, "best_logistic.pkl")

['best_logistic.pkl']

In [15]:

model = joblib.load("best_logistic.pkl")


In [16]:

import numpy as np

test_scores = model.predict_proba(X_test_vec)

# إذا رجعت list، نحولها إلى array
if isinstance(test_scores, list):
    test_scores = np.array([p[:, 1] for p in test_scores]).T

In [17]:
for k in [1, 2, 3, 4]:
    p, r, f = evaluate_at_k(y_test_bin, test_scores, k)

    print(f"\n===== Top-{k} Test Metrics =====")
    print(f"Precision@{k}: {p:.4f}")
    print(f"Recall@{k}:    {r:.4f}")
    print(f"F1@{k}:        {f:.4f}")


===== Top-1 Test Metrics =====
Precision@1: 0.7665
Recall@1:    0.6741
F1@1:        0.7173

===== Top-2 Test Metrics =====
Precision@2: 0.4632
Recall@2:    0.8148
F1@2:        0.5907

===== Top-3 Test Metrics =====
Precision@3: 0.3300
Recall@3:    0.8707
F1@3:        0.4786

===== Top-4 Test Metrics =====
Precision@4: 0.2564
Recall@4:    0.9021
F1@4:        0.3993


lineasvc

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier

# 1) TF-IDF
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=300000,
    sublinear_tf=True,
    binary=True
)

# ⚠️ fit فقط على train
X_train_vec = vectorizer.fit_transform(X_train)

# transform للباقي
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

print("شكل X_train:", X_train_vec.shape)

شكل X_train: (1493166, 300000)


In [19]:
joblib.dump(vectorizer, "tfidf_vectorizersvc.pkl")

['tfidf_vectorizersvc.pkl']

In [20]:
import joblib
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, jaccard_score

model = OneVsRestClassifier(
    LinearSVC(
        C=1,
        max_iter=10000,
        random_state=42
    ),
    n_jobs=-1
)

model.fit(X_train_vec, y_train_bin)



,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LinearSVC(C=1...ndom_state=42)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",-1
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the 

In [22]:
val_scores = model.decision_function(X_val_vec)
val_scores = np.asarray(val_scores)
print("val_scores shape:", val_scores.shape)


val_scores shape: (186646, 50)


In [23]:
def top_k_binary_predictions(y_scores, k):
    y_pred = np.zeros_like(y_scores, dtype=int)
    topk_idx = np.argsort(-y_scores, axis=1)[:, :k]

    for i in range(y_scores.shape[0]):
        y_pred[i, topk_idx[i]] = 1

    return y_pred

In [24]:
def evaluate_at_k(y_true, y_scores, k):
    y_pred_k = top_k_binary_predictions(y_scores, k)

    precision = precision_score(y_true, y_pred_k, average="micro", zero_division=0)
    recall = recall_score(y_true, y_pred_k, average="micro", zero_division=0)
    f1 = f1_score(y_true, y_pred_k, average="micro", zero_division=0)

    return precision, recall, f1

In [25]:
for k in [1, 2, 3, 4]:
    p, r, f = evaluate_at_k(y_val_bin, val_scores, k)

    print(f"\n===== Top-{k} Metrics =====")
    print(f"Precision@{k}: {p:.4f}")
    print(f"Recall@{k}:    {r:.4f}")
    print(f"F1@{k}:        {f:.4f}")


===== Top-1 Metrics =====
Precision@1: 0.7699
Recall@1:    0.6781
F1@1:        0.7211

===== Top-2 Metrics =====
Precision@2: 0.4587
Recall@2:    0.8079
F1@2:        0.5851

===== Top-3 Metrics =====
Precision@3: 0.3244
Recall@3:    0.8569
F1@3:        0.4706

===== Top-4 Metrics =====
Precision@4: 0.2513
Recall@4:    0.8851
F1@4:        0.3914


In [26]:
model = joblib.load("best_svc.pkl")


In [27]:
from sklearn.metrics import precision_score, recall_score, f1_score

# 1) استخراج scores من LinearSVC على test
test_scores = model.decision_function(X_test_vec)
test_scores = np.asarray(test_scores)

In [28]:
for k in [1, 2, 3, 4]:
    p, r, f = evaluate_at_k(y_test_bin, test_scores, k)

    print(f"\n===== Top-{k} Test Metrics =====")
    print(f"Precision@{k}: {p:.4f}")
    print(f"Recall@{k}:    {r:.4f}")
    print(f"F1@{k}:        {f:.4f}")


===== Top-1 Test Metrics =====
Precision@1: 0.7686
Recall@1:    0.6760
F1@1:        0.7193

===== Top-2 Test Metrics =====
Precision@2: 0.4585
Recall@2:    0.8066
F1@2:        0.5847

===== Top-3 Test Metrics =====
Precision@3: 0.3249
Recall@3:    0.8574
F1@3:        0.4713

===== Top-4 Test Metrics =====
Precision@4: 0.2518
Recall@4:    0.8858
F1@4:        0.3921
